# Person-Specific Sepsis Detection -- Production Pipeline
**Version:** v3 (Sensitivity Enhanced)

### Scoring Architecture
The system uses a **Hybrid Weighted Fusion** approach to combine multiple signals:
- **ML Probability (20%):** Patterns recognized by a Random Forest.
- **qSOFA Clinical Rules (30%):** Classical medical thresholds (HR, RR, SpO2).
- **Anomaly Detection (30%):** Uses an **L2-Norm Z-score** for high sensitivity to single-vital spikes.
- **Trajectory & Correlation (20%):** Detection of physiological decoupling and acceleration.

### Phase Definitions
| Phase | Duration | What happens |
|---|---|---|
| **Phase A -- Baseline** | T=0 -> T=200s (5 × 40s windows) | Collect vitals, compute 4-component confidence, lock unique baseline |
| **Phase B -- Monitoring** | T=200s -> ∞ (every 40s) | Continuous Z-scoring, anomaly detection, and sepsis phase classification |


In [18]:
import json
import time
import logging
import sys
import os

# --- ENVIRONMENT DIAGNOSTIC ---
print("Kernel Current Working Directory:", os.getcwd())
print("Looking for Project Files...")

# Search for project root
potential_roots = [
    os.getcwd(),
    "/mnt/c/Users/sreek/OneDrive/Desktop/SEPSIS_PERSON_SPECIFIC",
    "C:/Users/sreek/OneDrive/Desktop/SEPSIS_PERSON_SPECIFIC"
]

found_path = None
for p in potential_roots:
    if os.path.exists(os.path.join(p, "vitals_types.py")):
        if p not in sys.path:
            sys.path.append(p)
        found_path = p
        break

if found_path:
    print(f"SUCCESS: Found project modules at {found_path}")
else:
    print("WARNING: Could not find project files! Please ensure .py files are in the same folder as this notebook.")
    print("Search Path attempted:", potential_roots)
    print("Actual Directory Contents:", os.listdir())

# -----------------------------

from IPython.display import clear_output
from vitals_types import VitalsSample, BASELINE_WINDOWS
from simulator import PatientStreamSimulator
from models_factory import build_population_if, build_random_forest
from sepsis_detector import SepsisDetector

print("Building shared models (population IF + Random Forest)...")
pop_if = build_population_if()
rf     = build_random_forest()
print("Ready.")


Kernel Current Working Directory: c:\Users\sreek\OneDrive\Desktop\SEPSIS_PERSON_SPECIFIC
Looking for Project Files...
SUCCESS: Found project modules at c:\Users\sreek\OneDrive\Desktop\SEPSIS_PERSON_SPECIFIC
Building shared models (population IF + Random Forest)...
Ready.


In [19]:
# Load and display model performance metrics
metrics_path = "models/metrics.json"
if os.path.exists(metrics_path):
    with open(metrics_path, 'r') as f:
        m = json.load(f)
    print("=== Model Performance Metrics ===")
    print(f"Accuracy:  {m['accuracy']:.4f}")
    print(f"Precision: {m['precision']:.4f}")
    print(f"Recall:    {m['recall']:.4f}")
    print(f"F1-Score:  {m['f1']:.4f}")
else:
    print("Note: Performance metrics not found. Run 'python train_and_save_models.py' to generate them.")


=== Model Performance Metrics ===
Accuracy:  0.9611
Precision: 0.9619
Recall:    0.9611
F1-Score:  0.9611


In [20]:
# 0=normal  1=mild infection  2=sepsis (rapid deterioration)
BASELINE_CONDITION = 0

sim      = PatientStreamSimulator(condition=BASELINE_CONDITION)
detector = SepsisDetector(population_if=pop_if, rf_model=rf)
print(f"Initialised with baseline condition={BASELINE_CONDITION}.")


Initialised with baseline condition=0.


## Phase A -- Baseline Establishment
Collect 5 × 40-second windows. After window 5 the baseline **auto-locks**
and prints the confidence score and mode selection.


In [21]:
print("=== PHASE A: Collecting 5-window Baseline ===\n")
for _ in range(BASELINE_WINDOWS):
    sample = sim.get_next_window()
    result = detector.add_baseline_window(sample)
    if result:
        print("\n--- BASELINE LOCKED ---")
        print(json.dumps({
            "confidence":           result.confidence,
            "mode":                 result.mode,
            "confidence_breakdown": result.confidence_breakdown,
            "baseline_means":       {k: round(v, 2) for k, v in result.baseline_means.items()},
        }, indent=2))


=== PHASE A: Collecting 5-window Baseline ===


--- BASELINE LOCKED ---
{
  "confidence": 96.0,
  "mode": "LOCKED",
  "confidence_breakdown": {
    "Stability": 90.0,
    "Consistency": 100.0,
    "Activity": 100.0,
    "Variability": 100.0
  },
  "baseline_means": {
    "hr": 75.4,
    "rr": 14.29,
    "spo2": 97.88,
    "temp": 36.74,
    "movement": 10.67,
    "hrv": 44.91,
    "rrv": 13.89
  }
}


## Phase B -- Real-Time Monitoring
Runs 20 windows of simulated monitoring (condition=2 / rapid sepsis).
Set `DEMO_WINDOWS = None` for an infinite loop.


In [ ]:
DEMO_WINDOWS = None   # Run indefinitely until stopped manually (Kernel -> Interrupt)
sim.set_condition(2)

print("=== PHASE B: Real-Time Monitoring (condition=2 / sepsis) ===\n")
idx = 0
try:
    while DEMO_WINDOWS is None or idx < DEMO_WINDOWS:
        out = detector.process_monitoring_window(sim.get_next_window())
        clear_output(wait=True)
        print(f"Window {out['window_number']} | Status: {out['status']} | Phase: {out['sepsis_phase']}")
        print(f"Score: {out['final_score']:.3f} | Conf: {out['baseline_confidence']:.1f}% [{out['baseline_state']}]")
        print(f"qSOFA: {out['qsofa_score']} | HRV collapse: {out['hrv_collapse_severity']:.2f}")
        print()
        print(json.dumps(out, indent=2))
        time.sleep(0.8)
        idx += 1
except KeyboardInterrupt:
    print("\nMonitoring stopped.")


Window 3257 | Status: CRITICAL | Phase: PHASE_3_SEPTIC_SHOCK
Score: 1.000 | Conf: 96.0% [LOCKED]
qSOFA: 3 | HRV collapse: 0.00

{
  "phase": "MONITORING",
  "window_number": 3257,
  "timestamp": "2026-04-19T06:41:29.325238",
  "baseline_state": "LOCKED",
  "baseline_confidence": 96.0,
  "baseline_confidence_breakdown": {
    "Stability": 90.0,
    "Consistency": 100.0,
    "Activity": 100.0,
    "Variability": 100.0
  },
  "drift_from_locked": {
    "hr": 0.0598,
    "rr": 0.0134,
    "spo2": -0.0113,
    "temp": 0.0033,
    "movement": -0.0078,
    "hrv": -0.0287,
    "rrv": -0.02
  },
  "artifact_contaminated": false,
  "derivatives_available": true,
  "temp_bidirectional_flag": true,
  "vitals_current": {
    "timestamp": "2026-04-19T06:41:29.325238",
    "hr": 119.86,
    "rr": 25.78,
    "spo2": 88.34,
    "temp": 39.31,
    "movement": 7.55,
    "hrv": 12.83,
    "rrv": 4.97,
    "label": 2
  },
  "z_scores": {
    "hr": 81.847,
    "rr": 74.313,
    "spo2": -44.169,
    "temp": 

## Tuning Confidence Thresholds

Edit the constants at the top of `sepsis_detector_v2.py`:
```python
CONFIDENCE_HIGH = 75.0   # >= this -> LOCKED
CONFIDENCE_MID  = 60.0   # < this -> FALLBACK; between -> HYBRID
```

Run formal test suite to validate all 6 test cases:
```bash
python test_sepsis_v2.py
```
